In this notebook, the metric score results are aggregated into one file per metric

In [113]:
import pickle
import re

from src.utils.utils_data_formatter import save_dict_to_text

Starting with Complexity for Winequality data

In [114]:
def load_results(pickle_path):
    """Load dict from pickle file. 
    """
    with open(pickle_path, 'rb') as f:
        return pickle.load(f)

load the results obtained by running evaluate_mnist_dropout.py/evaluate_mnist_dropconnect.py

In [115]:
dropout_02 = load_results("/home/emilyschiller/dev/ua-eval/uncertainty_attribution_eval/results/mnist/2026-03-12/evaluation/cross_validation/100_samples/dropout/layers1/drop_prob0.2/cross_val_metrics_mnist.pkl")

In [116]:
dropconnect_01_ig = load_results("/home/emilyschiller/dev/ua-eval/uncertainty_attribution_eval/results/mnist/2026-03-13/09-20-00/evaluation/cross_validation/100_samples/dropconnect/layers_1/drop_prob0.1/cross_val_metrics_mnist 1.pkl")
dropconnect_01_shap = load_results("/home/emilyschiller/dev/ua-eval/uncertainty_attribution_eval/results/mnist/2026-03-12/09-31-05/evaluation/cross_validation/100_samples/dropconnect/layers_1/drop_prob0.1/cross_val_metrics_mnist.pkl")

In [117]:
import copy
from typing import Dict, Any

def merge_xai_dicts(*dicts: Dict[str, Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    """
    Merge mehrere nested dicts der Form
      { fold: { xai_method: {...}, ... }, ... }
    zu einem dict mit allen vorhandenen XAI-Methoden pro Fold.
    Bei Schlüsselkonflikten wird der spätere dict-Parameter verwendet.
    Werte werden deep-copied, damit keine Referenzen geteilt werden.
    """
    merged: Dict[str, Dict[str, Any]] = {}
    for src in dicts:
        for fold, methods in src.items():
            if fold not in merged:
                merged[fold] = {}
            for method_name, content in methods.items():
                merged[fold][method_name] = copy.deepcopy(content)
    return merged

In [118]:
dropconnect_01 = merge_xai_dicts(dropconnect_01_ig, dropconnect_01_shap)

In [119]:
dropconnect_01

{'fold_1': {'Integrated Gradients': {'Complexity': {'mean': 3.825914736732857,
    'std': 0.38227952317727115,
    'all_values': array([3.71865022, 4.08222282, 3.93520204, 3.12249965, 4.12207881,
           4.08668858, 3.73507725, 3.84836742, 3.87703459, 4.24439585,
           3.82156845, 4.30497034, 4.00208082, 3.46698299, 3.75163358,
           4.11260523, 4.30751869, 3.32496542, 3.71036666, 3.85681793,
           3.69168888, 4.11603664, 3.73240625, 3.41771253, 4.16928152,
           4.20018293, 4.05020475, 3.82102013, 3.74930853, 3.75279626,
           3.99308291, 4.12551564, 3.75553842, 3.46956598, 4.144755  ,
           4.56311127, 3.81111947, 4.14883549, 4.15260846, 3.59451525,
           2.8917179 , 3.78682297, 3.67812843, 3.8927551 , 4.51947128,
           4.0752227 , 3.67725208, 4.33424233, 3.91175387, 3.82228209,
           3.77913204, 2.86761948, 3.09834757, 3.96620982, 4.56896061,
           3.85882776, 3.89157698, 4.11439478, 3.63695688, 3.37435252,
           3.29942568, 

In [120]:
def aggregate_cross_val_dicts(named_dicts, metric="Complexity"):
    """
    Aggregate multiple cross-validation dicts into the desired format.

    Args:
      named_dicts (dict): Mapping name -> cross-validation-dict
          Example key: "dropconnect_005" or "dropout_01"
          Value: dict with keys "fold_1", "fold_2",
      metric (str): Name of the metric, default "Complexity".

    Returns:
      dict: Aggregated dict with structure
        {uq_name: { xai_method: {"mean":[...], "std":[...], "all_values":[ [...], [...], ... ] } } }
    """
    out = {}
    for name, cv_dict in named_dicts.items():
        out[name] = {}
        # Sort folds (fold_1, fold_2, ...) — numerically by the digit
        folds = sorted(
            (k for k in cv_dict.keys()),
            key=lambda s: int(re.search(r"\d+", s).group()) if re.search(r"\d+", s) else s # type: ignore
        )

        for fold in folds:
            methods = cv_dict.get(fold, {})
            if not isinstance(methods, dict):
                continue
            for xai_name, metrics_map in methods.items():
                if not isinstance(metrics_map, dict):
                    continue
                metric_block = metrics_map.get(metric)
                if not metric_block:
                    # if the requested metric is missing for this xai/fold, skip
                    continue

                if xai_name not in out[name]:
                    out[name][xai_name] = {"mean": [], "std": [], "all_values": []}

                # Ensure missing values become None instead of raising an error
                mean_val = metric_block.get("mean")
                std_val = metric_block.get("std")
                all_vals = metric_block.get("all_values")

                out[name][xai_name]["mean"].append(mean_val)
                out[name][xai_name]["std"].append(std_val)
                out[name][xai_name]["all_values"].append(all_vals)

    return out

In [121]:
named = {
    "dropout_02": dropout_02,
    "dropconnect_01": dropconnect_01,
}

aggregated = aggregate_cross_val_dicts(named, metric="Complexity")
save_dict_to_text(aggregated, save_dir= "/home/emilyschiller/dev/ua-eval/uncertainty_attribution_eval/results_final_mnist/Complexity", file_name ="complexity_aggregated")
out_path = "/home/emilyschiller/dev/ua-eval/uncertainty_attribution_eval/results_final_mnist/Complexity/complexity_aggregated.pkl"
with open(out_path, "wb") as f:
    pickle.dump(aggregated, f, protocol=pickle.HIGHEST_PROTOCOL)


Repeatability

In [122]:
def aggregate_cross_val_dicts_cosine_spearman(named_dicts, metric="Repeatability"):
    """
    Aggregate multiple cross-validation dicts into the desired format.

    Args:
      named_dicts (dict): Mapping name -> cross-validation-dict
          Example key: "dropconnect_005" or "dropout_01"
          Value: dict with keys "fold_1", "fold_2",
      metric (str): Name of the metric, default "Complexity".

    Returns:
      dict: Aggregated dict with structure
        {uq_name: { xai_method: {"mean":[...], "std":[...], "all_values":[ [...], [...], ... ] } } }
    """
    out = {}
    for name, cv_dict in named_dicts.items():
        out[name] = {}
        # Sort folds (fold_1, fold_2, ...) — numerically by the digit
        folds = sorted(
            (k for k in cv_dict.keys()),
            key=lambda s: int(re.search(r"\d+", s).group()) if re.search(r"\d+", s) else s # type: ignore
        )

        for fold in folds:
            methods = cv_dict.get(fold, {})
            if not isinstance(methods, dict):
                continue
            for xai_name, metrics_map in methods.items():
                if not isinstance(metrics_map, dict):
                    continue
                metric_block = metrics_map.get(metric)
                if not metric_block:
                    # if the requested metric is missing for this xai/fold, skip
                    continue

                if xai_name not in out[name]:
                    out[name][xai_name] = {"mean_cosine": [], "std_cosine": [], "mean_spearman": [], "std_spearman": [],"all_values_cosine": [], "all_values_spearman": []}

                # Ensure missing values become None instead of raising an error
                mean_val_cosine = metric_block.get("mean_cosine")
                std_val_cosine = metric_block.get("std_cosine")
                mean_val_spearman = metric_block.get("mean_spearman")
                std_val_spearman = metric_block.get("std_spearman")
                all_vals = metric_block.get("all_values")
                all_vals_cosine = all_vals[1]
                all_vals_spearman = all_vals[0]

                out[name][xai_name]["mean_cosine"].append(mean_val_cosine)
                out[name][xai_name]["std_cosine"].append(std_val_cosine)
                out[name][xai_name]["mean_spearman"].append(mean_val_spearman)
                out[name][xai_name]["std_spearman"].append(std_val_spearman)

                out[name][xai_name]["all_values_cosine"].append(all_vals_cosine)
                out[name][xai_name]["all_values_spearman"].append(all_vals_spearman)

    return out

In [123]:
named = {
    "dropout_02": dropout_02,
    "dropconnect_01": dropconnect_01,
}

aggregated = aggregate_cross_val_dicts_cosine_spearman(named, metric="Repeatability")
save_dict_to_text(aggregated, save_dir= "/home/emilyschiller/dev/ua-eval/uncertainty_attribution_eval/results_final_mnist/Determinism", file_name ="determinism_aggregated")
out_path = "/home/emilyschiller/dev/ua-eval/uncertainty_attribution_eval/results_final_mnist/Determinism/determinism_aggregated.pkl"
with open(out_path, "wb") as f:
    pickle.dump(aggregated, f, protocol=pickle.HIGHEST_PROTOCOL)

Feature Flipping

In [124]:
def aggregate_cross_val_dicts_featureflipping(named_dicts, metric="FeatureFlipping"):
    """
    Aggregate multiple cross-validation dicts into the desired format.

    Args:
      named_dicts (dict): Mapping name -> cross-validation-dict
          Example key: "dropconnect_005" or "dropout_01"
          Value: dict with keys "fold_1", "fold_2",
      metric (str): Name of the metric, default "Complexity".

    Returns:
      dict: Aggregated dict with structure
        {uq_name: { xai_method: {"mean":[...], "std":[...], "all_values":[ [...], [...], ... ] } } }
    """
    out = {}
    for name, cv_dict in named_dicts.items():
        out[name] = {}
        # Sort folds (fold_1, fold_2, ...) — numerically by the digit
        folds = sorted(
            (k for k in cv_dict.keys()),
            key=lambda s: int(re.search(r"\d+", s).group()) if re.search(r"\d+", s) else s # type: ignore
        )

        for fold in folds:
            methods = cv_dict.get(fold, {})
            if not isinstance(methods, dict):
                continue
            for xai_name, metrics_map in methods.items():
                if not isinstance(metrics_map, dict):
                    continue
                metric_block = metrics_map.get(metric)
                if not metric_block:
                    # if the requested metric is missing for this xai/fold, skip
                    continue

                if xai_name not in out[name]:
                    out[name][xai_name] = {"auc_mean": [], "auc_stds": [], "all_values": []}

                # Ensure missing values become None instead of raising an error
                mean_val = metric_block.get("auc_mean")
                std_val = metric_block.get("auc_stds")
                all_vals = metric_block.get("all_values")

                out[name][xai_name]["auc_mean"].append(mean_val)
                out[name][xai_name]["auc_stds"].append(std_val)
                out[name][xai_name]["all_values"].append(all_vals)

    return out

In [ ]:
named = {
    "dropout_02": dropout_02,
    "dropconnect_01": dropconnect_01,
}

aggregated = aggregate_cross_val_dicts_featureflipping(named, metric="FeatureFlipping")
save_dict_to_text(aggregated, save_dir= "/home/emilyschiller/dev/ua-eval/uncertainty_attribution_eval/results_final_mnist/FeatureFlipping", file_name ="feature_flipping_aggregated")
out_path = "/home/emilyschiller/dev/ua-eval/uncertainty_attribution_eval/results_final_mnist/FeatureFlipping/feature_flipping_aggregated.pkl"
with open(out_path, "wb") as f:
    pickle.dump(aggregated, f, protocol=pickle.HIGHEST_PROTOCOL)

Relative Input Stability

In [128]:
def aggregate_cross_val_dicts_ris(named_dicts, metric="RelativeInputStability"):
    """
    Aggregate multiple cross-validation dicts into the desired format.

    Args:
      named_dicts (dict): Mapping name -> cross-validation-dict
          Example key: "dropconnect_005" or "dropout_01"
          Value: dict with keys "fold_1", "fold_2",
      metric (str): Name of the metric, default "Complexity".

    Returns:
      dict: Aggregated dict with structure
        {uq_name: { xai_method: {"mean":[...], "std":[...], "all_values":[ [...], [...], ... ] } } }
    """
    out = {}
    for name, cv_dict in named_dicts.items():
        out[name] = {}
        # Sort folds (fold_1, fold_2, ...) — numerically by the digit
        folds = sorted(
            (k for k in cv_dict.keys()),
            key=lambda s: int(re.search(r"\d+", s).group()) if re.search(r"\d+", s) else s # type: ignore
        )

        for fold in folds:
            methods = cv_dict.get(fold, {})
            if not isinstance(methods, dict):
                continue
            for xai_name, metrics_map in methods.items():
                if not isinstance(metrics_map, dict):
                    continue
                metric_block = metrics_map.get(metric)
                if not metric_block:
                    # if the requested metric is missing for this xai/fold, skip
                    continue

                if xai_name not in out[name]:
                    out[name][xai_name] = {"RIS_mean": [], "RIS_std": [], "all_values": [], "nr_perturbations": []}

                # Ensure missing values become None instead of raising an error
                mean_val = metric_block.get("RIS_mean")
                std_val = metric_block.get("RIS_std")
                all_list = metric_block.get("all_values")
                all_vals = all_list[0]
                nr_perturbations = all_list[1]

                out[name][xai_name]["RIS_mean"].append(mean_val)
                out[name][xai_name]["RIS_std"].append(std_val)
                out[name][xai_name]["all_values"].append(all_vals)
                out[name][xai_name]["nr_perturbations"].append(nr_perturbations)
    return out

In [129]:
named = {
    "dropout_02": dropout_02,
    "dropconnect_01": dropconnect_01,
}

aggregated = aggregate_cross_val_dicts_ris(named, metric="RelativeInputStability")
save_dict_to_text(aggregated, save_dir= "/home/emilyschiller/dev/ua-eval/uncertainty_attribution_eval/results_final_mnist/RelativeInputStability", file_name ="relative_input_stability_aggregated")
out_path = "/home/emilyschiller/dev/ua-eval/uncertainty_attribution_eval/results_final_mnist/RelativeInputStability/relative_input_stability_aggregated.pkl"
with open(out_path, "wb") as f:
    pickle.dump(aggregated, f, protocol=pickle.HIGHEST_PROTOCOL)

Relative Rank Improvement

In [130]:
def aggregate_cross_val_dicts_rri(named_dicts, metric="Relative Rank Improvement"):
    """
    Aggregate multiple cross-validation dicts into the desired format.

    Args:
      named_dicts (dict): Mapping name -> cross-validation-dict
          Example key: "dropconnect_005" or "dropout_01"
          Value: dict with keys "fold_1", "fold_2",
      metric (str): Name of the metric, default "Complexity".

    Returns:
      dict: Aggregated dict with structure
        {uq_name: { xai_method: {"mean":[...], "std":[...], "all_values":[ [...], [...], ... ] } } }
    """
    out = {}
    for name, cv_dict in named_dicts.items():
        out[name] = {}
        # Sort folds (fold_1, fold_2, ...) — numerically by the digit
        folds = sorted(
            (k for k in cv_dict.keys()),
            key=lambda s: int(re.search(r"\d+", s).group()) if re.search(r"\d+", s) else s # type: ignore
        )

        for fold in folds:
            methods = cv_dict.get(fold, {})
            if not isinstance(methods, dict):
                continue
            for xai_name, metrics_map in methods.items():
                if not isinstance(metrics_map, dict):
                    continue
                metric_block = metrics_map.get(metric)
                if not metric_block:
                    # if the requested metric is missing for this xai/fold, skip
                    continue

                if xai_name not in out[name]:
                    out[name][xai_name] = {"avg_unc_rank_change": [], "accuracy": [], "ranks": [], "rank_changes": []}

                # Ensure missing values become None instead of raising an error
                rank_change = metric_block.get("avg_unc_rank_change")
                acc = metric_block.get("accuracy")
                all_list = metric_block.get("all_values")
                ranks = all_list[0]
                rank_changes = all_list[1]

                out[name][xai_name]["avg_unc_rank_change"].append(rank_change)
                out[name][xai_name]["accuracy"].append(acc)
                out[name][xai_name]["ranks"].append(ranks)
                out[name][xai_name]["rank_changes"].append(rank_changes)
    return out

In [131]:
named = {
    "dropout_02": dropout_02,
    "dropconnect_01": dropconnect_01,
}

aggregated = aggregate_cross_val_dicts_rri(named, metric="Relative Rank Improvement")
save_dict_to_text(aggregated, save_dir= "/home/emilyschiller/dev/ua-eval/uncertainty_attribution_eval/results_final_mnist/RelativeRankImprovement", file_name ="relative_rank_improvement_aggregated")
out_path = "/home/emilyschiller/dev/ua-eval/uncertainty_attribution_eval/results_final_mnist/RelativeRankImprovement/relative_rank_improvement_aggregated.pkl"
with open(out_path, "wb") as f:
    pickle.dump(aggregated, f, protocol=pickle.HIGHEST_PROTOCOL)

Uncertainty Conveyance Similarity

In [132]:
def aggregate_cross_val_dicts_ucs(named_dicts, metric="Uncertainty Conveyance Similarity"):
    """
    Aggregate multiple cross-validation dicts into the desired format.

    Args:
      named_dicts (dict): Mapping name -> cross-validation-dict
          Example key: "dropconnect_005" or "dropout_01"
          Value: dict with keys "fold_1", "fold_2",
      metric (str): Name of the metric, default "Complexity".

    Returns:
      dict: Aggregated dict with structure
        {uq_name: { xai_method: {"mean":[...], "std":[...], "all_values":[ [...], [...], ... ] } } }
    """
    out = {}
    for name, cv_dict in named_dicts.items():
        out[name] = {}
        # Sort folds (fold_1, fold_2, ...) — numerically by the digit
        folds = sorted(
            (k for k in cv_dict.keys()),
            key=lambda s: int(re.search(r"\d+", s).group()) if re.search(r"\d+", s) else s # type: ignore
        )

        for fold in folds:
            methods = cv_dict.get(fold, {})
            if not isinstance(methods, dict):
                continue
            for xai_name, metrics_map in methods.items():
                if not isinstance(metrics_map, dict):
                    continue
                metric_block = metrics_map.get(metric)
                if not metric_block:
                    # if the requested metric is missing for this xai/fold, skip
                    continue

                if xai_name not in out[name]:
                    out[name][xai_name] = {"mean_cosine": [], "std_cosine": [], "mean_spearman": [], "std_spearman": [],"all_values_cosine": [], "all_values_spearman": []}

                # Ensure missing values become None instead of raising an error
                mean_val_cosine = metric_block.get("cosine_mean")
                std_val_cosine = metric_block.get("cosine_std")
                mean_val_spearman = metric_block.get("spearmanr_mean")
                std_val_spearman = metric_block.get("spearmanr_std")
                all_vals = metric_block.get("all_values")
                all_vals_cosine = all_vals[1]
                all_vals_spearman = all_vals[0]

                out[name][xai_name]["mean_cosine"].append(mean_val_cosine)
                out[name][xai_name]["std_cosine"].append(std_val_cosine)
                out[name][xai_name]["mean_spearman"].append(mean_val_spearman)
                out[name][xai_name]["std_spearman"].append(std_val_spearman)

                out[name][xai_name]["all_values_cosine"].append(all_vals_cosine)
                out[name][xai_name]["all_values_spearman"].append(all_vals_spearman)

    return out

In [133]:
named = {
    "dropout_02": dropout_02,
    "dropconnect_01": dropconnect_01,
}

aggregated = aggregate_cross_val_dicts_ucs(named, metric="Uncertainty Conveyance Similarity")
save_dict_to_text(aggregated, save_dir= "/home/emilyschiller/dev/ua-eval/uncertainty_attribution_eval/results_final_mnist/UncertaintyConveyanceSimilarity", file_name ="uncertainty_conveyance_similarity_aggregated")
out_path = "/home/emilyschiller/dev/ua-eval/uncertainty_attribution_eval/results_final_mnist/UncertaintyConveyanceSimilarity/uncertainty_conveyance_similarity_aggregated.pkl"
with open(out_path, "wb") as f:
    pickle.dump(aggregated, f, protocol=pickle.HIGHEST_PROTOCOL)